# 02 — Training the fusion model

Trains the architecture of README §3.2: **shared per-modality encoders** (audio, IMU) for each board, and a **fusion head**. Then it exports the three parts as separate int8 `.tflite` files for MicroFlow.

```mermaid
flowchart LR
    subgraph A["Board A"]
        aA["audio_A<br/>(1, 800, 1)"] --> EA["audio encoder<br/>(shared)"] --> zA["32-d"]
        lA["loge_A<br/>(1)"]
        iA["imu_A<br/>(1, 100, 6)"] --> IA["IMU encoder<br/>(shared)"] --> ziA["16-d"]
    end
    subgraph B["Board B"]
        aB["audio_B"] --> EB["audio encoder<br/>(same weights)"] --> zB["32-d"]
        lB["loge_B"]
        iB["imu_B"] --> IB["IMU encoder<br/>(same weights)"] --> ziB["16-d"]
    end
    zA & lA & ziA & zB & lB & ziB --> C["concat<br/>(in Rust on the MCU)"] --> H["head<br/>Dense 32 → Dense n_zones"] --> Y["zone"]
```

## Joint training or frozen encoders?

**Recommendation: train everything jointly (end to end), with two safeguards.** Then freeze the encoders only for the cases where freezing actually pays off.

Why joint training:
- The encoders learn the features the *fusion* needs. For instance, the audio encoder can focus on what the IMU does not capture.
- The dataset is small and the same for every branch, so a separate pre-training stage would see exactly the same data. It brings nothing but an extra stage.

The risk of joint training is **modality competition**: the dominant branch (probably the microphone) gets most of the gradient and the IMU branch stays undertrained. The two safeguards:
1. **Auxiliary heads**: one small classifier per branch, with a loss term weighted by 0.3. This forces every encoder to be discriminative on its own. A bonus is that each branch can then be evaluated alone.
2. **Branch dropout**: during training, a whole branch is zeroed at random. The head learns to cope with a missing modality, or with a missing board (a late BLE message). This matches the late-fusion fallback planned in the risks section.

When freezing the encoders *is* the right call (shown in section 6):
- **Calibration on a new table or a new session** with a few taps per zone: retrain only the head. With 45 examples, fine-tuning all 12 k parameters overfits.
- **Comparing fusion strategies** (split vs. late fusion over BLE) with identical encoders, so the comparison is fair.
- **Compression stages**: compress one part at a time and measure each part separately.

Freezing also makes head training almost free: the embeddings are computed once, and training a ~3 k-parameter MLP on them takes seconds. It could even run on the Raspberry Pi.

> ⚠️ With `DATA = "synthetic"`, the numbers below only check that the pipeline works. They say nothing about a real table.

## 1. Configuration

In [ ]:
import os

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")  # silence TensorFlow's C++ logs (before importing it)
from pathlib import Path

import keras
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

from touchwood import dataset as D
from touchwood import model as M
from touchwood.export import TFLiteModel, check_microflow, to_int8_tflite

DATA = "synthetic"  # "synthetic" until real recordings exist, then "recordings"
RECORDINGS = Path("../data/raw")
EXPORT_DIR = Path("../../firmware/inference/models")

WIDTH = 1.0  # width multiplier: 1.0 = student; ~4.0 gives a teacher for distillation
EPOCHS = 80
BATCH = 32
AUX_WEIGHT = 0.3  # weight of each auxiliary (per-branch) loss
BRANCH_DROPOUT = 0.15  # probability of zeroing a whole branch during training
SEED = 0
keras.utils.set_random_seed(SEED)

## 2. Data

In [ ]:
if DATA == "synthetic":
    raw, labels = D.synthetic(n_per_zone=100, sessions=2, seed=SEED)
else:
    raw, labels = D.load_recordings(RECORDINGS)

BOARDS = tuple(sorted(k.split("_")[1] for k in raw if k.startswith("audio_")))
USE_IMU = all(f"imu_{b}" in raw for b in BOARDS)  # the firmware does not stream the IMU yet
N_CLASSES = len(labels)
y_all = raw["y"]
print(f"{len(y_all)} taps, zones={labels}, boards={BOARDS}, IMU={USE_IMU}, sessions={sorted(set(raw['group']))}")

In [ ]:
# One tap per zone as seen by each board: the fingerprints should differ between zones.
fig, axes = plt.subplots(len(BOARDS), min(N_CLASSES, 5), figsize=(15, 2.2 * len(BOARDS)), sharex=True, squeeze=False)
t_ms = (np.arange(M.AUDIO_LEN) - D.PRE) / D.FS * 1e3
for r, b in enumerate(BOARDS):
    for c in range(axes.shape[1]):
        axes[r, c].plot(t_ms, raw[f"audio_{b}"][np.flatnonzero(y_all == c)[0]], lw=0.6)
        axes[r, c].set_title(f"board {b}, {labels[c]}", fontsize=9)
plt.tight_layout()

## 3. Split

- **Test = the last recording session**, when there are several. It measures generalisation to a *new session*: another day, the board slightly moved. This is the honest number. A random split mixes taps from the same session and is optimistic.
- Validation = 15 % of the remaining taps, stratified, used for early stopping.

In [ ]:
groups = raw["group"]
sessions = sorted(set(groups))
if len(sessions) > 1:
    test_idx = np.flatnonzero(groups == sessions[-1])
    rest = np.flatnonzero(groups != sessions[-1])
else:
    rest, test_idx = train_test_split(np.arange(len(y_all)), test_size=0.2, stratify=y_all, random_state=SEED)
train_idx, val_idx = train_test_split(rest, test_size=0.15, stratify=y_all[rest], random_state=SEED)

# IMU scale: one global constant from the training set (to be hard-coded in the firmware).
IMU_SCALE = float(np.std(np.concatenate([raw[f"imu_{b}"][train_idx] for b in BOARDS]))) if USE_IMU else 1.0
X = D.prepare(raw, BOARDS, USE_IMU, IMU_SCALE)
X_train, X_val, X_test = (D.subset(X, i) for i in (train_idx, val_idx, test_idx))
y_train, y_val, y_test = y_all[train_idx], y_all[val_idx], y_all[test_idx]
print(
    f"train {len(train_idx)} / val {len(val_idx)} / test {len(test_idx)} (test session: {sessions[-1]}), IMU_SCALE={IMU_SCALE:.4g}"
)

## 4. Model and joint training

In [ ]:
model, parts = M.build_fusion_model(BOARDS, USE_IMU, N_CLASSES, width=WIDTH, branch_dropout=BRANCH_DROPOUT)
aux = [layer.name for layer in model.layers if layer.name.startswith("aux_")]
OUTPUTS = ["zone", *aux]  # keys of the model's output dict
model.compile(
    optimizer=keras.optimizers.Adam(3e-3),
    loss={name: "sparse_categorical_crossentropy" for name in OUTPUTS},
    loss_weights={"zone": 1.0, **{name: AUX_WEIGHT for name in aux}},
    metrics={"zone": ["accuracy"]},
)
for name, part in parts.items():
    print(f"{name:14s} {part.count_params():6d} params (exported)")
print(f"{'aux heads':14s} {sum(model.get_layer(n).count_params() for n in aux):6d} params (training only)")

In [ ]:
targets = lambda y: {name: y for name in OUTPUTS}
history = model.fit(
    X_train,
    targets(y_train),
    validation_data=(X_val, targets(y_val)),
    epochs=EPOCHS,
    batch_size=BATCH,
    verbose=0,
    callbacks=[
        keras.callbacks.EarlyStopping("val_zone_accuracy", patience=15, restore_best_weights=True, mode="max"),
        keras.callbacks.ReduceLROnPlateau("val_loss", factor=0.5, patience=6),
    ],
)
h = history.history
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].plot(h["loss"], label="train")
axes[0].plot(h["val_loss"], label="val")
axes[0].set(title="total loss", xlabel="epoch")
axes[1].plot(h["zone_accuracy"], label="train")
axes[1].plot(h["val_zone_accuracy"], label="val")
axes[1].set(title="zone accuracy", xlabel="epoch")
for ax in axes:
    ax.legend()
print(f"stopped after {len(h['loss'])} epochs, best val accuracy {max(h['val_zone_accuracy']):.1%}")

## 5. Evaluation and ablations

The head is evaluated **exactly as the firmware will run it**: encoders first, then the concatenation outside the graph, then the head. Ablations zero branches the same way branch dropout did during training:
- *board B missing* is the fallback when its BLE message is late;
- *mic only* and *IMU only* tell what each modality contributes.

In [ ]:
def accuracy(drop=(), X_=X_test, y_=y_test):
    p = parts["head"].predict(M.features(parts, X_, BOARDS, USE_IMU, drop), verbose=0)
    return (p.argmax(1) == y_).mean(), p.argmax(1)


acc, pred = accuracy()
print(f"test accuracy (fusion): {acc:.1%}")
ConfusionMatrixDisplay.from_predictions(
    y_test, pred, display_labels=labels, normalize="true", cmap="Blues", values_format=".2f"
)
plt.title("Test session")

ablations = {"all branches": ()}
if len(BOARDS) > 1:
    for b in BOARDS:
        others = [o for o in BOARDS if o != b]
        ablations[f"board {b} only"] = tuple(f"{m}_{o}" for o in others for m in ("audio", "imu"))
if USE_IMU:
    ablations["mic only"] = tuple(f"imu_{b}" for b in BOARDS)
    ablations["IMU only"] = tuple(f"audio_{b}" for b in BOARDS)
for name, drop in ablations.items():
    print(f"  {name:15s} {accuracy(drop)[0]:.1%}")

## 6. When freezing the encoders pays off

Three variants, all evaluated on the test session:

| Variant | Trains | Data |
|---|---|---|
| (a) fresh head, frozen encoders | the head only (~3 k params) | the full training set, on precomputed embeddings |
| (b) **calibration**, frozen encoders | the head only, fine-tuned | **K taps per zone from the new session** |
| (c) calibration, everything unfrozen | the whole model (~12 k params) | the same K taps per zone |

The expected outcome: (a) roughly matches joint training, so the joint stage did not need freezing. (b) recovers most of the session drift. (c) tends to overfit or forget, because 12 k parameters is a lot for `K × n_zones` examples.

In [ ]:
K = 5  # calibration taps per zone, taken from the test session
rng = np.random.default_rng(SEED)
calib_idx = np.concatenate([rng.choice(np.flatnonzero(y_test == c), K, replace=False) for c in range(N_CLASSES)])
eval_idx = np.setdiff1d(np.arange(len(y_test)), calib_idx)
X_cal, y_cal = D.subset(X_test, calib_idx), y_test[calib_idx]
X_ev, y_ev = D.subset(X_test, eval_idx), y_test[eval_idx]

F_train = M.features(parts, X_train, BOARDS, USE_IMU)  # embeddings computed once
F_cal, F_ev = (M.features(parts, x, BOARDS, USE_IMU) for x in (X_cal, X_ev))


def train_head(head, F, y, epochs, lr):
    head.compile(keras.optimizers.Adam(lr), "sparse_categorical_crossentropy", metrics=["accuracy"])
    head.fit(F, y, epochs=epochs, batch_size=BATCH, verbose=0)
    return (head.predict(F_ev, verbose=0).argmax(1) == y_ev).mean()


results = {"joint model, no calibration": (parts["head"].predict(F_ev, verbose=0).argmax(1) == y_ev).mean()}

# (a) fresh head trained on frozen embeddings of the training set
results["(a) fresh head on frozen encoders"] = train_head(
    M.build_head(F_train.shape[1], N_CLASSES), F_train, y_train, 60, 3e-3
)

# (b) calibration: copy of the trained head, fine-tuned on K taps/zone of the new session
head_b = M.build_head(F_train.shape[1], N_CLASSES)
head_b.set_weights(parts["head"].get_weights())
results[f"(b) calibration, frozen encoders (K={K})"] = train_head(head_b, F_cal, y_cal, 40, 1e-3)

# (c) calibration with everything unfrozen (on a clone, to keep the original model intact)
model_c = keras.models.clone_model(model)
model_c.set_weights(model.get_weights())
model_c.compile(
    keras.optimizers.Adam(1e-3),
    loss={n: "sparse_categorical_crossentropy" for n in OUTPUTS},
    loss_weights={"zone": 1.0, **{n: AUX_WEIGHT for n in aux}},
)
model_c.fit(X_cal, {n: y_cal for n in OUTPUTS}, epochs=40, batch_size=BATCH, verbose=0)
results[f"(c) calibration, all unfrozen (K={K})"] = (model_c.predict(X_ev, verbose=0)["zone"].argmax(1) == y_ev).mean()

for name, acc in results.items():
    print(f"{name:45s} {acc:.1%}")

## 7. Export for MicroFlow (int8, one model per part)

Each part is rebuilt with a static batch size of 1, quantized to int8 with calibration data from the training set, and checked against MicroFlow's operator set. Then the **int8 pipeline** (two encoders, concatenation, head) is evaluated on the test session, to measure the cost of quantization.

Constants the firmware must reproduce: per-board audio normalisation by the window peak, `loge = log10(peak + 1) / LOGE_SCALE`, the IMU divided by `IMU_SCALE`, and the order of the head input `[audio_b (32), loge_b (1), imu_b (16)]` for each board in turn.

In [ ]:
n_cal = min(300, len(train_idx))
exports = {
    "audio_encoder": (
        M.for_export(parts["audio_encoder"], M.build_audio_encoder, width=WIDTH),
        np.concatenate([X_train[f"audio_{b}"][:n_cal] for b in BOARDS]),
    ),
    "head": (M.for_export(parts["head"], M.build_head, in_dim=F_train.shape[1], n_classes=N_CLASSES), F_train[:n_cal]),
}
if USE_IMU:
    exports["imu_encoder"] = (
        M.for_export(parts["imu_encoder"], M.build_imu_encoder, width=WIDTH),
        np.concatenate([X_train[f"imu_{b}"][:n_cal] for b in BOARDS]),
    )

tflite = {}
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
for name, (part, calib) in exports.items():
    tflite[name] = to_int8_tflite(part, calib)
    ops = check_microflow(tflite[name])
    (EXPORT_DIR / f"{name}.tflite").write_bytes(tflite[name])
    print(f"{name:14s} {part.count_params():6d} params  {len(tflite[name]) / 1024:5.1f} KB  ops={sorted(ops)}")
print(f"total {sum(map(len, tflite.values())) / 1024:.1f} KB -> {EXPORT_DIR.resolve()}")
print(f"firmware constants: LOGE_SCALE={D.LOGE_SCALE}, IMU_SCALE={IMU_SCALE:.6g}")

In [ ]:
# int8 pipeline on the PC, one sample at a time, the way the MCU will run it.
enc_a = TFLiteModel(tflite["audio_encoder"])
enc_i = TFLiteModel(tflite["imu_encoder"]) if USE_IMU else None
head_q = TFLiteModel(tflite["head"])
cols = []
for b in BOARDS:
    cols += [enc_a(X_test[f"audio_{b}"]), X_test[f"loge_{b}"]]
    if USE_IMU:
        cols.append(enc_i(X_test[f"imu_{b}"]))
acc_q = (head_q(np.concatenate(cols, 1)).argmax(1) == y_test).mean()
print(f"test accuracy: float {accuracy()[0]:.1%} -> int8 {acc_q:.1%}")

## 8. Next steps

- [ ] Real data: set `DATA = "recordings"` once `01_feasibility` says go.
- [ ] Teacher: `WIDTH = 4`, more epochs, then distil into the student (notebook 03): temperature and α sweep, like TP4.
- [ ] Compression (notebook 04): structured pruning, SVD on the head's Dense layers, clustering, QAT. Freeze what you are not compressing.
- [ ] Firmware: load the three models in `firmware/inference`, apply the constants above, and send the 49-d per-board features over BLE.
- [ ] Measure on the device: latency per part, flash, RAM, energy.